In [ ]:
#@title Setup (run me first!) { display-mode: "form" }
#@markdown This cell loads the shared infrastructure, generates data, and
#@markdown precomputes all interactive elements. Takes ~20 seconds.

import json, os, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
warnings.filterwarnings('ignore')

# ── Color palette ──
COLORS = {
    'ols': '#2E5EA8',
    'truth': '#D4A017',
    'bias': '#C0392B',
    'repair': '#27AE60',
    'alt': '#7F8C8D',
}

# ── Gate System ──
_trap_responses = {}
_TRAP_LOG_PATH = '/content/trap_log.json'

def _load_trap_log():
    global _trap_responses
    if os.path.exists(_TRAP_LOG_PATH):
        try:
            with open(_TRAP_LOG_PATH, 'r') as f:
                _trap_responses = json.load(f)
        except (json.JSONDecodeError, IOError):
            _trap_responses = {}

def _save_trap_log():
    try:
        os.makedirs(os.path.dirname(_TRAP_LOG_PATH), exist_ok=True)
        with open(_TRAP_LOG_PATH, 'w') as f:
            json.dump(_trap_responses, f, indent=2)
    except IOError:
        pass

def record_trap_response(notebook_id, question_id, response):
    key = f"{notebook_id}_{question_id}"
    _trap_responses[key] = {"response": response, "timestamp": datetime.now().isoformat()}
    _save_trap_log()

def get_trap_response(notebook_id, question_id):
    key = f"{notebook_id}_{question_id}"
    entry = _trap_responses.get(key)
    return entry["response"] if entry else None

def check_gate(notebook_id, question_id):
    return f"{notebook_id}_{question_id}" in _trap_responses

def get_all_responses():
    return dict(_trap_responses)

def create_trap_widget(notebook_id, question_id, question_text, options):
    label = widgets.HTML(f"<b>{question_text}</b>")
    radio = widgets.RadioButtons(options=options, value=None, layout=widgets.Layout(width='auto'))
    submit = widgets.Button(description="Submit", button_style='primary')
    output = widgets.Output()
    def on_submit(btn):
        with output:
            output.clear_output()
            if radio.value is None:
                print("Please select an option before submitting.")
                return
            record_trap_response(notebook_id, question_id, radio.value)
            print(f"Response recorded: {radio.value}")
            submit.disabled = True
            radio.disabled = True
    submit.on_click(on_submit)
    existing = get_trap_response(notebook_id, question_id)
    if existing is not None:
        try:
            radio.value = existing
            radio.disabled = True
            submit.disabled = True
        except Exception:
            pass
    return widgets.VBox([label, radio, submit, output])

_load_trap_log()

# ── CompanySimulator ──
class CompanySimulator:
    def __init__(self, endogeneity_strength=20, heteroscedasticity_strength=0.5,
                 nonlinearity=True, bad_control_strength=0.1, noise_sigma=1.0):
        self.endogeneity_strength = endogeneity_strength
        self.heteroscedasticity_strength = heteroscedasticity_strength
        self.nonlinearity = nonlinearity
        self.bad_control_strength = bad_control_strength
        self.noise_sigma = noise_sigma
        self.beta_0 = 50
        self.beta_1 = 8
        self.beta_2 = 3
        self.beta_U = 2
        self.staff_loading = 5

    def generate(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        return pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2, 'satisfaction': X3})

    def truth(self, n=500, seed=42):
        rng = np.random.default_rng(seed)
        U = rng.standard_normal(n)
        eta1 = rng.normal(0, self.noise_sigma, n)
        eta2 = rng.normal(0, self.noise_sigma, n)
        eta3 = rng.normal(0, self.noise_sigma, n)
        X1 = self.endogeneity_strength * U + eta1
        if self.nonlinearity:
            X1 = np.abs(X1) + 0.01
        X2 = self.staff_loading * U + eta2
        eps_std = np.sqrt(np.abs(self.heteroscedasticity_strength * X1))
        eps = rng.normal(0, eps_std)
        if self.nonlinearity:
            Y = self.beta_0 + self.beta_1 * np.log(X1) + self.beta_2 * X2 + self.beta_U * U + eps
        else:
            Y = self.beta_0 + self.beta_1 * X1 + self.beta_2 * X2 + self.beta_U * U + eps
        X3 = self.bad_control_strength * Y + eta3
        df = pd.DataFrame({'revenue': Y, 'ad_spend': X1, 'staff_count': X2,
                           'satisfaction': X3, 'demand_U': U})
        params = {'beta_0': self.beta_0, 'beta_1': self.beta_1, 'beta_2': self.beta_2,
                  'beta_U': self.beta_U, 'sigma_epsilon': self.heteroscedasticity_strength}
        return df, params

    def dgp_summary(self):
        func = r"\log(X_1)" if self.nonlinearity else r"X_1"
        return (rf"$Y = {self.beta_0} + {self.beta_1} \cdot {func} "
                rf"+ {self.beta_2} \cdot X_2 + {self.beta_U} \cdot U + \varepsilon$")

    def set_heteroscedasticity(self, strength):
        self.heteroscedasticity_strength = strength
    def set_endogeneity(self, strength):
        self.endogeneity_strength = strength
    def set_nonlinearity(self, curvature):
        self.nonlinearity = bool(curvature)

# ── MonteCarloEngine ──
class MonteCarloEngine:
    def run(self, estimator_fn, param_name, param_grid, simulator, n_reps=5000, n_obs=500):
        results = np.empty((len(param_grid), n_reps))
        try:
            progress = widgets.IntProgress(value=0, min=0, max=len(param_grid),
                                           description='Simulating:', style={'description_width': 'initial'})
            display(progress)
            use_widget = True
        except Exception:
            use_widget = False
        setter = getattr(simulator, f'set_{param_name}', None)
        for i, val in enumerate(param_grid):
            if setter is not None:
                setter(val)
            for r in range(n_reps):
                data = simulator.generate(n=n_obs, seed=r)
                results[i, r] = estimator_fn(data)
            if use_widget:
                progress.value = i + 1
        if use_widget:
            progress.bar_style = 'success'
        return results

    def quick_run(self, estimator_fn, dgp_fn, n_reps=1000, n_obs=500):
        results = np.empty(n_reps)
        for r in range(n_reps):
            data = dgp_fn(seed=r, n=n_obs)
            results[r] = estimator_fn(data)
        return results

# ── DiagnosticSuite ──
class DiagnosticSuite:
    @staticmethod
    def run_diagnostics(model_result):
        fig, axes = plt.subplots(2, 2, figsize=(10, 8))
        influence = OLSInfluence(model_result)
        fitted = model_result.fittedvalues
        resid = model_result.resid
        std_resid = influence.resid_studentized_internal
        ax = axes[0, 0]
        ax.scatter(fitted, resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1.5)
        ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals'); ax.set_title('Residuals vs Fitted')
        ax = axes[0, 1]
        stats.probplot(resid, dist="norm", plot=ax)
        ax.set_title('Normal Q-Q')
        ax.get_lines()[0].set_color(COLORS['ols'])
        ax.get_lines()[1].set_color(COLORS['bias'])
        ax = axes[1, 0]
        sqrt_abs_std = np.sqrt(np.abs(std_resid))
        ax.scatter(fitted, sqrt_abs_std, alpha=0.4, s=15, color=COLORS['ols'])
        ax.set_xlabel('Fitted values')
        ax.set_ylabel(r'$\sqrt{|Standardized\ Residuals|}$')
        ax.set_title('Scale-Location')
        ax = axes[1, 1]
        leverage = influence.hat_matrix_diag
        ax.scatter(leverage, std_resid, alpha=0.4, s=15, color=COLORS['ols'])
        ax.axhline(0, color=COLORS['truth'], linewidth=1)
        ax.set_xlabel('Leverage'); ax.set_ylabel('Standardized Residuals')
        ax.set_title('Residuals vs Leverage')
        fig.tight_layout()
        return fig

    @staticmethod
    def summary_tests(model_result):
        results = {}
        exog = model_result.model.exog
        exog_names = model_result.model.exog_names
        vif_dict = {}
        for i, name in enumerate(exog_names):
            if name == 'const': continue
            vif_dict[name] = variance_inflation_factor(exog, i)
        results['vif'] = vif_dict
        bp_stat, bp_pval, _, _ = het_breuschpagan(model_result.resid, model_result.model.exog)
        results['breusch_pagan'] = (bp_stat, bp_pval)
        results['durbin_watson'] = durbin_watson(model_result.resid)
        jb_stat, jb_pval, _, _ = jarque_bera(model_result.resid)
        results['jarque_bera'] = (jb_stat, jb_pval)
        return results

# ── AutopsyReport ──
class AutopsyReport:
    @staticmethod
    def lightweight(notebook_number):
        threat = widgets.Text(description='Biggest threat:', placeholder='What is the biggest threat to this estimate?',
                              layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        check = widgets.Text(description='How to check:', placeholder='How would you check if that threat is real?',
                             layout=widgets.Layout(width='90%'), style={'description_width': '120px'})
        submit = widgets.Button(description='Save Autopsy', button_style='primary')
        output = widgets.Output()
        def on_submit(btn):
            with output:
                output.clear_output()
                nb_id = f"notebook_{notebook_number}"
                record_trap_response(nb_id, "threat", threat.value)
                record_trap_response(nb_id, "check", check.value)
                print("Autopsy responses saved.")
                submit.disabled = True
        submit.on_click(on_submit)
        return widgets.VBox([widgets.HTML(f"<h3>Mini Autopsy \u2014 Notebook {notebook_number}</h3>"),
                             threat, check, submit, output])

# ── Generate data for this notebook ──
sim = CompanySimulator()
df = sim.generate(n=500, seed=42)
df_truth, true_params = sim.truth(n=500, seed=42)

# Fit the linear model (the trap)
X_lin = sm.add_constant(df['ad_spend'])
model_linear = sm.OLS(df['revenue'], X_lin).fit()

# Fit the log model (the repair)
X_log = sm.add_constant(np.log(df['ad_spend']))
model_log = sm.OLS(df['revenue'], X_log).fit()

# True function for plotting
def true_revenue(ad_spend_val):
    """Expected revenue at a given ad_spend, averaging over U and noise."""
    return sim.beta_0 + sim.beta_1 * np.log(ad_spend_val)

# Precompute extrapolation cache for slider
x_max = df['ad_spend'].max()
x_min = df['ad_spend'].min()
extrap_positions = np.concatenate([
    np.linspace(x_min, x_max, 11),
    np.linspace(x_max * 1.5, x_max * 10, 10)
])
extrap_positions = np.round(extrap_positions, 1)

print("Setup complete!")
print(f"Data: {len(df)} observations of NovaMart monthly data.")
print(f"Ad spend range: ${df['ad_spend'].min()*1000:.0f} to ${df['ad_spend'].max()*1000:.0f}")

# Notebook 4: Why Your Model Is Wrong

*Your model is an approximation. Inside the range of your data, it might be fine. Outside that range, it's fiction with a confidence interval.*

---

You've been doing solid work at NovaMart. You controlled for confounders (Notebook 1), fixed your standard errors (Notebook 2), and learned to distinguish statistical from practical significance (Notebook 3).

Now the VP of Marketing wants a forecast. She's considering a **massive** ad campaign — $2 million — far larger than anything NovaMart has ever tried. She wants to know: **how much revenue would that generate?**

You have a model. It has R² = 0.71. Let's use it.

In [ ]:
# ── The Trap: Linear model looks great ──

print(model_linear.summary())
print(f"\nR-squared: {model_linear.rsquared:.2f}")
print(f"\nLinear prediction at $2M ad spend: ${model_linear.predict([1, 2000])[0]*1000:,.0f}")

In [ ]:
# ── The Trap: Your prediction ──
# The VP wants to know: how much revenue from a $2M campaign?
# Your linear model says each unit of ad_spend yields revenue.
# Enter your prediction below.

trap_label = widgets.HTML(
    "<b>The VP asks: 'How much monthly revenue would a $2M ad campaign generate?'</b><br>"
    "<i>Your linear model has R\u00b2=0.71. Ad spend is in thousands. "
    "Enter your prediction in thousands of dollars.</i>"
)

prediction_input = widgets.FloatText(
    value=0,
    description='Prediction ($K):',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='300px')
)

# Pre-fill with linear model prediction as a hint
linear_pred_2M = model_linear.predict([1, 2000])[0]
prediction_input.value = round(linear_pred_2M, 1)

submit_btn = widgets.Button(description='Submit Prediction', button_style='primary')
trap_output = widgets.Output()

def on_trap_submit(btn):
    with trap_output:
        trap_output.clear_output()
        val = prediction_input.value
        record_trap_response('notebook_4', 'extrapolation_prediction', val)
        print(f"Prediction recorded: ${val*1000:,.0f} monthly revenue from a $2M campaign.")
        submit_btn.disabled = True
        prediction_input.disabled = True

submit_btn.on_click(on_trap_submit)

# Pre-fill if already answered
existing = get_trap_response('notebook_4', 'extrapolation_prediction')
if existing is not None:
    prediction_input.value = existing
    prediction_input.disabled = True
    submit_btn.disabled = True

display(widgets.VBox([trap_label, prediction_input, submit_btn, trap_output]))

In [ ]:
# ── The Reveal: True relationship vs. linear fit ──

if not check_gate('notebook_4', 'extrapolation_prediction'):
    print("\u26a0\ufe0f Please submit your prediction in the cell above before continuing.")
else:
    student_pred = get_trap_response('notebook_4', 'extrapolation_prediction')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left panel: inside data range
    x_range_in = np.linspace(x_min, x_max, 200)
    ax1.scatter(df['ad_spend'], df['revenue'], alpha=0.3, s=15, color=COLORS['alt'],
                label='Observed data')
    ax1.plot(x_range_in, model_linear.predict(sm.add_constant(x_range_in)),
             color=COLORS['ols'], linewidth=2, linestyle='-', label='Linear fit')
    ax1.plot(x_range_in, true_revenue(x_range_in),
             color=COLORS['truth'], linewidth=2.5, linestyle='-', label='True relationship (log)')
    ax1.set_xlabel('Ad Spend ($K)', fontsize=12)
    ax1.set_ylabel('Revenue ($K)', fontsize=12)
    ax1.set_title('Inside data range: nearly identical', fontsize=13)
    ax1.legend(fontsize=10)

    # Right panel: extrapolation to $2M
    x_range_out = np.linspace(x_min, 2000, 500)
    linear_preds = model_linear.predict(sm.add_constant(x_range_out))
    true_preds = true_revenue(x_range_out)

    ax2.scatter(df['ad_spend'], df['revenue'], alpha=0.2, s=10, color=COLORS['alt'],
                label='Observed data')
    ax2.plot(x_range_out, linear_preds, color=COLORS['ols'], linewidth=2,
             linestyle='-', label='Linear extrapolation')
    ax2.plot(x_range_out, true_preds, color=COLORS['truth'], linewidth=2.5,
             linestyle='-', label='True relationship (log)')

    # Mark $2M point
    lin_at_2M = model_linear.predict([1, 2000])[0]
    true_at_2M = true_revenue(2000)
    ax2.plot(2000, lin_at_2M, 'o', color=COLORS['ols'], markersize=10, zorder=5)
    ax2.plot(2000, true_at_2M, 'D', color=COLORS['truth'], markersize=10, zorder=5)
    if student_pred is not None:
        ax2.plot(2000, student_pred, '^', color=COLORS['bias'], markersize=12,
                 zorder=6, label=f'Your prediction: ${student_pred*1000:,.0f}')

    # Data range boundary
    ax2.axvline(x_max, color=COLORS['alt'], linestyle=':', alpha=0.7, label=f'Data boundary ({x_max:.0f}K)')
    ax2.axvspan(x_max, 2100, alpha=0.08, color=COLORS['bias'])

    ax2.set_xlabel('Ad Spend ($K)', fontsize=12)
    ax2.set_ylabel('Revenue ($K)', fontsize=12)
    ax2.set_title('Beyond data range: wild divergence', fontsize=13)
    ax2.legend(fontsize=9, loc='upper left')

    fig.suptitle('Specification Error: Linear Model vs. True Logarithmic Relationship',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Quantify the error
    overshoot = lin_at_2M / true_at_2M
    print(f"Linear prediction at $2M:  ${lin_at_2M*1000:>12,.0f}")
    print(f"True expected value at $2M: ${true_at_2M*1000:>12,.0f}")
    print(f"Overshoot ratio: {overshoot:.1f}x")
    if student_pred is not None:
        student_overshoot = student_pred / true_at_2M
        print(f"\nYour prediction: ${student_pred*1000:,.0f} ({student_overshoot:.1f}x the truth)")
        if student_overshoot > 1.5:
            print("\nYou fell into the extrapolation trap. The linear model made it look reasonable.")
        elif student_overshoot < 0.8:
            print("\nInteresting \u2014 you were more conservative than the model. Good instinct.")
        else:
            print("\nYou were close! But without knowing the true functional form, how could you have known?")

## The Problem: Specification Error

Your model assumed a **linear** relationship between ad spend and revenue. The true relationship is **logarithmic** — each additional dollar of ad spend has diminishing returns.

Inside your data range, both functions look nearly identical. That's why R² = 0.71 — a straight line is a decent local approximation of a curve. But beyond your data, they diverge wildly.

---

**The theorem behind this:**

> If the true model is $Y = f(X) + \varepsilon$ and we fit $Y = X\beta + \varepsilon$, OLS converges to the **linear projection coefficient** — the best linear approximation — not $f'(x)$. The approximation is local. Extrapolation inherits the error of the functional form, not just the noise.

In plain English: **OLS finds the best straight line through a curve. Inside the data, it works. Outside, it's fiction.**

In [ ]:
# ── Monte Carlo Confirmation ──
# Show that specification error produces systematically wrong predictions outside data range

n_mc = 1000
pred_at_2M_linear = np.empty(n_mc)
pred_at_2M_true = np.empty(n_mc)

for r in range(n_mc):
    d = sim.generate(n=500, seed=r)
    X_mc = sm.add_constant(d['ad_spend'])
    m = sm.OLS(d['revenue'], X_mc).fit()
    pred_at_2M_linear[r] = m.predict([1, 2000])[0]
    pred_at_2M_true[r] = true_revenue(2000)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pred_at_2M_linear, bins=40, alpha=0.7, color=COLORS['ols'], edgecolor='white',
        label=f'Linear predictions (mean={np.mean(pred_at_2M_linear):.1f})')
ax.axvline(true_revenue(2000), color=COLORS['truth'], linewidth=3, linestyle='-',
           label=f'True value = {true_revenue(2000):.1f}')
ax.axvline(np.mean(pred_at_2M_linear), color=COLORS['bias'], linewidth=2, linestyle=':',
           label=f'Mean linear prediction = {np.mean(pred_at_2M_linear):.1f}')

bias = np.mean(pred_at_2M_linear) - true_revenue(2000)
ax.set_title(f'1,000 Linear Models Predicting at $2M | Systematic Overshoot = {bias:.1f}K',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Revenue ($K)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Every single linear model overshoots the truth.")
print(f"This isn't noise \u2014 it's systematic specification error.")

In [ ]:
# ── The Destruction Playground: Extrapolation slider ──
# Slider controls where you're predicting. Watch linear and true diverge.

slider_positions = np.concatenate([
    np.linspace(x_min, x_max, 15),
    np.linspace(x_max * 1.5, x_max * 8, 10)
])
slider_positions = np.round(slider_positions, 1)

# Precompute predictions and confidence bands
precomputed = {}
for pos in slider_positions:
    lin_pred = model_linear.predict([1, pos])[0]
    log_pred = true_revenue(pos)
    # Confidence interval for linear model
    pred_frame = pd.DataFrame({'const': [1], 'ad_spend': [pos]})
    try:
        lin_ci = model_linear.get_prediction(pred_frame).conf_int(alpha=0.05)[0]
    except Exception:
        se = np.sqrt(model_linear.mse_resid * (1 + 1/len(df) + (pos - df['ad_spend'].mean())**2 / np.sum((df['ad_spend'] - df['ad_spend'].mean())**2)))
        lin_ci = [lin_pred - 1.96*se, lin_pred + 1.96*se]
    precomputed[pos] = {
        'linear': lin_pred, 'true': log_pred,
        'ci_low': lin_ci[0], 'ci_high': lin_ci[1],
        'error': lin_pred - log_pred,
        'in_data': pos <= x_max
    }

slider = widgets.SelectionSlider(
    options=[(f'${v:.0f}K', v) for v in slider_positions],
    value=slider_positions[0],
    description='Predict at:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

output_plot = widgets.Output()
output_summary = widgets.Output()

def update_extrapolation(change):
    pos = change['new']
    p = precomputed[pos]

    with output_plot:
        output_plot.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5))

        # Plot data
        ax.scatter(df['ad_spend'], df['revenue'], alpha=0.2, s=10, color=COLORS['alt'])

        # Plot both lines across full range
        x_plot = np.linspace(x_min, max(pos * 1.1, x_max), 300)
        ax.plot(x_plot, model_linear.predict(sm.add_constant(x_plot)),
                color=COLORS['ols'], linewidth=2, linestyle='-',
                marker='o', markevery=len(x_plot), label='Linear prediction')
        ax.plot(x_plot, true_revenue(x_plot),
                color=COLORS['truth'], linewidth=2.5, linestyle='-',
                marker='D', markevery=len(x_plot), label='True (log) prediction')

        # Confidence band for linear
        x_ci = np.linspace(x_min, max(pos * 1.1, x_max), 50)
        ci_lo = []; ci_hi = []
        for xv in x_ci:
            se = np.sqrt(model_linear.mse_resid * (1 + 1/len(df) +
                         (xv - df['ad_spend'].mean())**2 /
                         np.sum((df['ad_spend'] - df['ad_spend'].mean())**2)))
            pred = model_linear.predict([1, xv])[0]
            ci_lo.append(pred - 1.96*se)
            ci_hi.append(pred + 1.96*se)
        ax.fill_between(x_ci, ci_lo, ci_hi, alpha=0.15, color=COLORS['ols'])

        # Mark prediction point
        ax.axvline(pos, color=COLORS['bias'], linewidth=2, linestyle=':', alpha=0.8)
        ax.plot(pos, p['linear'], 'o', color=COLORS['ols'], markersize=12, zorder=5)
        ax.plot(pos, p['true'], 'D', color=COLORS['truth'], markersize=12, zorder=5)

        # Data boundary
        ax.axvline(x_max, color=COLORS['alt'], linestyle='--', alpha=0.5)
        if pos > x_max:
            ax.axvspan(x_max, max(pos * 1.1, x_max * 1.1), alpha=0.06, color=COLORS['bias'])

        status = '(inside data)' if p['in_data'] else '(EXTRAPOLATION)'
        contains = 'YES' if p['ci_low'] <= p['true'] <= p['ci_high'] else 'NO'
        ax.set_title(f'Predicting at ${pos:.0f}K {status} | '
                     f'Error = {p["error"]:+.1f}K | CI contains truth: {contains}',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Ad Spend ($K)', fontsize=11)
        ax.set_ylabel('Revenue ($K)', fontsize=11)
        ax.legend(fontsize=10)
        plt.tight_layout()
        plt.show()

    with output_summary:
        output_summary.clear_output(wait=True)
        print(f"Prediction point:  ${pos:.0f}K ad spend")
        print(f"Linear prediction:  {p['linear']:.1f}K revenue")
        print(f"True expected:      {p['true']:.1f}K revenue")
        print(f"Error:              {p['error']:+.1f}K")
        print(f"95% CI:            [{p['ci_low']:.1f}, {p['ci_high']:.1f}]")
        print(f"CI contains truth:  {'Yes' if p['ci_low'] <= p['true'] <= p['ci_high'] else 'NO!'}")
        if not p['in_data']:
            ratio = pos / x_max
            print(f"\nExtrapolating {ratio:.1f}x beyond observed data!")

slider.observe(update_extrapolation, names='value')

# Initial render
update_extrapolation({'new': slider.value})

layout = widgets.HBox([
    widgets.VBox([slider, output_plot], layout=widgets.Layout(width='70%')),
    output_summary
], layout=widgets.Layout(width='100%'))
display(layout)

### Discussion

> *Your model has R² = 0.71 inside the data range. Your manager asks for a prediction at 10x the maximum observed X. What do you say?*

Think about this before continuing. There is no single correct answer — this is a judgment call that statistics alone cannot resolve.

In [ ]:
# ── The Repair: Log-transformed model ──

print("=" * 60)
print("REPAIR: Fitting log-transformed model")
print("=" * 60)
print()
print(model_log.summary())
print(f"\nLog model R-squared: {model_log.rsquared:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_extrap = np.linspace(x_min, 2000, 500)

# Left: inside data
x_in = np.linspace(x_min, x_max, 200)
ax1.scatter(df['ad_spend'], df['revenue'], alpha=0.2, s=10, color=COLORS['alt'])
ax1.plot(x_in, model_linear.predict(sm.add_constant(x_in)),
         color=COLORS['ols'], linewidth=2, linestyle='-', label='Linear model')
ax1.plot(x_in, model_log.predict(sm.add_constant(np.log(x_in))),
         color=COLORS['repair'], linewidth=2, linestyle='-.', marker='s', markevery=30,
         label='Log model (repair)')
ax1.plot(x_in, true_revenue(x_in),
         color=COLORS['truth'], linewidth=2.5, linestyle='-',
         label='True relationship')
ax1.set_title('Inside data: both models similar', fontsize=12)
ax1.set_xlabel('Ad Spend ($K)'); ax1.set_ylabel('Revenue ($K)')
ax1.legend(fontsize=9)

# Right: extrapolation
ax2.scatter(df['ad_spend'], df['revenue'], alpha=0.15, s=8, color=COLORS['alt'])
ax2.plot(x_extrap, model_linear.predict(sm.add_constant(x_extrap)),
         color=COLORS['ols'], linewidth=2, linestyle='-', label='Linear model')
ax2.plot(x_extrap, model_log.predict(sm.add_constant(np.log(x_extrap))),
         color=COLORS['repair'], linewidth=2, linestyle='-.', marker='s', markevery=60,
         label='Log model (repair)')
ax2.plot(x_extrap, true_revenue(x_extrap),
         color=COLORS['truth'], linewidth=2.5, linestyle='-',
         label='True relationship')
ax2.axvline(x_max, color=COLORS['alt'], linestyle='--', alpha=0.5, label='Data boundary')

# Predictions at $2M
log_at_2M = model_log.predict([1, np.log(2000)])[0]
true_at_2M = true_revenue(2000)
ax2.plot(2000, lin_at_2M, 'o', color=COLORS['ols'], markersize=10)
ax2.plot(2000, log_at_2M, 's', color=COLORS['repair'], markersize=10)
ax2.plot(2000, true_at_2M, 'D', color=COLORS['truth'], markersize=10)

ax2.set_title('Extrapolation: log model tracks truth', fontsize=12)
ax2.set_xlabel('Ad Spend ($K)'); ax2.set_ylabel('Revenue ($K)')
ax2.legend(fontsize=9)

fig.suptitle('Repair: Log-Transformed Model Recovers True Functional Form',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nPredictions at $2M ad spend:")
print(f"  Linear model: ${lin_at_2M*1000:>10,.0f}  (error: {lin_at_2M - true_at_2M:+.1f}K)")
print(f"  Log model:    ${log_at_2M*1000:>10,.0f}  (error: {log_at_2M - true_at_2M:+.1f}K)")
print(f"  Truth:        ${true_at_2M*1000:>10,.0f}")

In [ ]:
# ── The Limit: Even the log model fails at extreme extrapolation ──

limit_positions = np.array([100, 500, 1000, 2000, 5000, 10000, 20000, 50000])

# Precompute for limit slider
limit_cache = {}
for pos in limit_positions:
    log_pred = model_log.predict([1, np.log(pos)])[0]
    # At extreme values, the true DGP includes X2 and U terms that create
    # additional nonlinearity not captured by simple log(X1)
    # Simulate: at extreme ad_spend, demand U drives ad_spend,
    # so the conditional E[Y|X1=x] includes E[U|X1=x] which is nonlinear
    true_simple = true_revenue(pos)
    # The log model ignores that U is correlated with X1
    # At extreme X1, E[U|X1=x] = x/endogeneity_strength (approx)
    implied_U = pos / sim.endogeneity_strength
    implied_X2 = sim.staff_loading * implied_U
    true_conditional = sim.beta_0 + sim.beta_1 * np.log(pos) + sim.beta_2 * implied_X2 + sim.beta_U * implied_U
    limit_cache[pos] = {
        'log_pred': log_pred,
        'true_simple': true_simple,
        'true_conditional': true_conditional,
        'log_error': log_pred - true_conditional
    }

limit_slider = widgets.SelectionSlider(
    options=[(f'${v:,.0f}K', v) for v in limit_positions],
    value=limit_positions[0],
    description='Extrapolate to:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='80%'),
    continuous_update=False
)

limit_output = widgets.Output()

def update_limit(change):
    pos = change['new']
    with limit_output:
        limit_output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(11, 5))

        x_plot = np.linspace(x_min, pos * 1.05, 400)
        ax.scatter(df['ad_spend'], df['revenue'], alpha=0.15, s=8, color=COLORS['alt'])

        # Linear
        ax.plot(x_plot, model_linear.predict(sm.add_constant(x_plot)),
                color=COLORS['ols'], linewidth=1.5, linestyle=':', alpha=0.5,
                label='Linear (wrong)')
        # Log model
        ax.plot(x_plot, model_log.predict(sm.add_constant(np.log(x_plot))),
                color=COLORS['repair'], linewidth=2, linestyle='-.',
                marker='s', markevery=60, label='Log model (repair)')
        # True conditional
        true_cond = []
        for xv in x_plot:
            iu = xv / sim.endogeneity_strength
            ix2 = sim.staff_loading * iu
            true_cond.append(sim.beta_0 + sim.beta_1 * np.log(xv) + sim.beta_2 * ix2 + sim.beta_U * iu)
        ax.plot(x_plot, true_cond, color=COLORS['truth'], linewidth=2.5,
                linestyle='-', label='True conditional E[Y|X1]')

        ax.axvline(x_max, color=COLORS['alt'], linestyle='--', alpha=0.5)

        c = limit_cache[pos]
        ratio = pos / x_max
        ax.set_title(f'Extrapolation to ${pos:,.0f}K ({ratio:.0f}x data range) | '
                     f'Log model error: {c["log_error"]:+.1f}K',
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Ad Spend ($K)', fontsize=11)
        ax.set_ylabel('Revenue ($K)', fontsize=11)
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

        if pos >= 10000:
            print(f"\nAt ${pos:,.0f}K, even the log model misses by {c['log_error']:+.1f}K.")
            print("Every functional form is a local approximation.")
            print("The log model captures diminishing returns but ignores")
            print("that extreme ad spend implies extreme demand (U), which")
            print("also drives revenue through other channels.")

limit_slider.observe(update_limit, names='value')
update_limit({'new': limit_positions[0]})
display(widgets.VBox([limit_slider, limit_output]))

In [ ]:
#@title Real-World Disaster: The Challenger O-Ring Catastrophe { display-mode: "form" }

# ── Sidebar: Challenger disaster ──

story_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Story</h4>
<p>On January 28, 1986, the Space Shuttle Challenger broke apart 73 seconds after launch,
killing all seven crew members. The cause: O-ring failure in the solid rocket boosters.</p>

<p>The night before launch, engineers at Morton Thiokol argued against launching in the
forecasted 36\u00b0F weather. They had data on O-ring damage from previous flights. But when
they presented their analysis, they made a critical statistical error: <b>they only analyzed
flights that had O-ring damage.</b></p>

<p>Looking at only the damaged flights, there was no clear temperature trend. NASA managers
asked: "Where's the pattern?" The engineers couldn't show one. The launch proceeded.</p>

<p>Had they included <b>all 23 previous flights</b> \u2014 including those with no damage \u2014 the
pattern was unmistakable: every flight below 65\u00b0F had damage. The 36\u00b0F launch temperature
was far beyond any previous experience. Seven people died because of selection on the
dependent variable.</p>
</div>
""")

math_tab = widgets.HTML(value="""
<div style='padding: 15px; font-size: 13px; line-height: 1.6;'>
<h4>The Math: Selection on Y</h4>
<p>Let the true model be: $P(\\text{damage}) = f(\\text{temperature})$</p>

<p>If we only observe flights where damage = 1, we're conditioning on Y = 1.
This truncates the sample and destroys the relationship between X and Y.</p>

<p>Formally, by conditioning on $Y=1$:</p>
<ul>
<li>We lose all information from $Y=0$ observations (undamaged flights)</li>
<li>The conditional distribution $X | Y=1$ has different properties than $X$</li>
<li>The regression of $Y$ on $X$ using only $Y=1$ cases is biased toward zero</li>
</ul>

<p>This is also a specification error: the "model" of analyzing only failures
misspecifies the sample, which is equivalent to misspecifying the relationship.</p>
</div>
""")

sim_tab = widgets.Output()

def run_challenger_sim(btn):
    with sim_tab:
        sim_tab.clear_output(wait=True)
        rng = np.random.default_rng(42)

        # Simulate 23 flights
        n_flights = 23
        temps = rng.uniform(50, 82, n_flights)
        # True probability of damage increases as temp decreases
        prob_damage = 1 / (1 + np.exp(0.2 * (temps - 60)))
        damage = rng.binomial(1, prob_damage)
        # Add some noise to damage severity (1-3 scale)
        severity = damage * rng.integers(1, 4, n_flights)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

        # Left: Only flights with damage (the mistake)
        mask_damage = severity > 0
        ax1.scatter(temps[mask_damage], severity[mask_damage],
                    color=COLORS['bias'], s=80, marker='^', zorder=5,
                    label='Flights with damage')
        if mask_damage.sum() >= 2:
            X_dam = sm.add_constant(temps[mask_damage])
            m_dam = sm.OLS(severity[mask_damage], X_dam).fit()
            x_line = np.linspace(30, 85, 100)
            ax1.plot(x_line, m_dam.predict(sm.add_constant(x_line)),
                     color=COLORS['bias'], linestyle=':', linewidth=2,
                     label=f'Regression (p={m_dam.pvalues[1]:.2f})')

        ax1.axvline(36, color=COLORS['alt'], linestyle='--', linewidth=2,
                    label='Launch temp (36\u00b0F)')
        ax1.set_xlabel('Temperature (\u00b0F)', fontsize=11)
        ax1.set_ylabel('O-Ring Damage Index', fontsize=11)
        ax1.set_title('What NASA Saw: Failures Only\n(No clear trend)', fontsize=12)
        ax1.legend(fontsize=9)
        ax1.set_xlim(30, 85)

        # Right: All flights (the truth)
        ax2.scatter(temps[mask_damage], severity[mask_damage],
                    color=COLORS['bias'], s=80, marker='^', zorder=5,
                    label='Flights with damage')
        ax2.scatter(temps[~mask_damage], severity[~mask_damage],
                    color=COLORS['repair'], s=80, marker='s', zorder=5,
                    label='Flights without damage')

        X_all = sm.add_constant(temps)
        m_all = sm.OLS(severity, X_all).fit()
        ax2.plot(x_line, m_all.predict(sm.add_constant(x_line)),
                 color=COLORS['truth'], linewidth=2.5,
                 label=f'Regression (p={m_all.pvalues[1]:.3f})')

        ax2.axvline(36, color=COLORS['alt'], linestyle='--', linewidth=2,
                    label='Launch temp (36\u00b0F)')
        ax2.set_xlabel('Temperature (\u00b0F)', fontsize=11)
        ax2.set_ylabel('O-Ring Damage Index', fontsize=11)
        ax2.set_title('All 23 Flights: Clear Trend\n(Extrapolation to 36\u00b0F is terrifying)', fontsize=12)
        ax2.legend(fontsize=9)
        ax2.set_xlim(30, 85)

        fig.suptitle('Challenger O-Ring Disaster: Selection on Y Hid the Pattern',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

        print(f"Damage-only regression: slope = {m_dam.params[1]:.3f}, p = {m_dam.pvalues[1]:.3f}")
        print(f"All-flights regression: slope = {m_all.params[1]:.3f}, p = {m_all.pvalues[1]:.3f}")
        print(f"\nSelecting on Y=1 destroyed the signal. Seven people died.")

run_btn = widgets.Button(description='Run Simulation', button_style='primary')
run_btn.on_click(run_challenger_sim)

sim_content = widgets.VBox([run_btn, sim_tab])

tabs = widgets.Tab(children=[story_tab, math_tab, sim_content])
tabs.set_title(0, 'The Story')
tabs.set_title(1, 'The Math')
tabs.set_title(2, 'The Simulation')

display(tabs)

In [ ]:
# ── Mini Autopsy ──
display(AutopsyReport.lightweight(4))

---

## Bridge

*Your model is now correctly specified. But how do you know it's not just memorizing the data?*

A model that fits your existing data perfectly might predict terribly on new data. That's not a bug — it's the **bias-variance tradeoff**.

**Next: Notebook 5 — Why Your R² Is Lying**

In [ ]:
# ── Transition Card ──
display(HTML("""
<div style='background: linear-gradient(135deg, #1a1a2e, #16213e); padding: 30px;
            border-radius: 10px; margin: 20px 0; color: white; font-family: sans-serif;'>
    <p style='font-size: 14px; opacity: 0.8; margin: 0;'>NEXT IN THE SERIES</p>
    <h2 style='margin: 10px 0; font-size: 24px;'>Notebook 5: Why Your R\u00b2 Is Lying</h2>
    <p style='font-size: 16px; opacity: 0.9; margin: 10px 0;'>
        A model that fits your existing data perfectly will predict new data terribly.
        This is not a paradox \u2014 it's memorization vs. learning.
    </p>
    <a href='05_overfitting.ipynb' style='display: inline-block; background: #D4A017;
       color: #1a1a2e; padding: 10px 24px; border-radius: 5px; text-decoration: none;
       font-weight: bold; margin-top: 10px;'>Open Notebook 5 \u2192</a>
</div>
"""))